In [4]:
import pandas as pd

# load data
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

print(df.head())
print("Shape:", df.shape)

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
Sh

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# load data
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

# drop irrelevant columns
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
# features & target
X = df.drop("Survived", axis=1)
y = df["Survived"]

#types of column
num_cols = ["Age", "Fare", "SibSp", "Parch"]
cat_cols = ["Sex", "Embarked", "Pclass"]

#pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:

from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

##ans-4
random Forest selected cause it performs well on tabular datasets, handles non-linear relationships, reduces overfitting  and needs minimal feature scaling compared with the other models.

In [7]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked',
                                                   'Pclass'])])),
                ('model', RandomForestClassifier(random_state=42))])

In [8]:
from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("Cross-validation scores:", scores)
print("Mean:", np.mean(scores))
print("Std:", np.std(scores))

Cross-validation scores: [0.7972028  0.75524476 0.8028169  0.77464789 0.83802817]
Mean: 0.7935881020388063
Std: 0.027935999314557945


In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'model__max_depth': 5, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best Score: 0.8201713779178567


In [11]:
best_model = grid.best_estimator_

In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8100558659217877

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.90      0.85       105
           1       0.82      0.69      0.75        74

    accuracy                           0.81       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.81      0.81       179


Confusion Matrix:
 [[94 11]
 [23 51]]


In [17]:
import gradio as gr
import joblib
import pandas as pd

model = joblib.load("model.pkl")

def predict(pclass, sex, age, sibsp, parch, fare, embarked):
    data = pd.DataFrame([{
        "Pclass": pclass,
        "Sex": sex,
        "Age": age,
        "SibSp": sibsp,
        "Parch": parch,
        "Fare": fare,
        "Embarked": embarked
    }])

    prediction = model.predict(data)[0]
    return "Survived" if prediction == 1 else "Did Not Survive"

interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Pclass"),
        gr.Radio(["male", "female"], label="Sex"),
        gr.Number(label="Age"),
        gr.Number(label="SibSp"),
        gr.Number(label="Parch"),
        gr.Number(label="Fare"),
        gr.Radio(["C", "Q", "S"], label="Embarked")
    ],
    outputs="text"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b580ae3a35253022c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
import joblib
joblib.dump(best_model, "model.pkl")

['model.pkl']

In [18]:
import os
print(os.listdir())

['.config', 'model.pkl', '.gradio', 'sample_data']


In [26]:
!pip install huggingface_hub -q

In [27]:
from huggingface_hub import notebook_login
notebook_login()

In [29]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "Imran-Nayem/Titanic-Passanger-survial-Prediction"

api.upload_file(
    path_or_fileobj="model.pkl",
    path_in_repo="model.pkl",
    repo_id=repo_id,
    repo_type="space"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  model.pkl                   : 100%|##########|  807kB /  807kB            

  model.pkl                   : 100%|##########|  807kB /  807kB            

CommitInfo(commit_url='https://huggingface.co/spaces/Imran-Nayem/Titanic-Passanger-survial-Prediction/commit/2cb6e8cd1e946dba5ce039a9cbe6e6437ae2c584', commit_message='Upload model.pkl with huggingface_hub', commit_description='', oid='2cb6e8cd1e946dba5ce039a9cbe6e6437ae2c584', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/Imran-Nayem/Titanic-Passanger-survial-Prediction', endpoint='https://huggingface.co', repo_type='space', repo_id='Imran-Nayem/Titanic-Passanger-survial-Prediction'), pr_revision=None, pr_num=None)

In [36]:
app_code = """
import gradio as gr
import joblib
import pandas as pd

model = joblib.load("model.pkl")

def predict(pclass, sex, age, sibsp, parch, fare, embarked):
    data = pd.DataFrame([{
        "Pclass": int(pclass),
        "Sex": sex,
        "Age": float(age),
        "SibSp": int(sibsp),
        "Parch": int(parch),
        "Fare": float(fare),
        "Embarked": embarked
    }])

    try:
        pred = model.predict(data)[0]
        return "Survived" if pred == 1 else "Did Not Survive"
    except Exception as e:
        return "Error: " + str(e)

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Pclass", value=3),
        gr.Radio(["male", "female"], label="Sex", value="male"),
        gr.Number(label="Age", value=25),
        gr.Number(label="SibSp", value=0),
        gr.Number(label="Parch", value=0),
        gr.Number(label="Fare", value=7.25),
        gr.Radio(["C", "Q", "S"], label="Embarked", value="S")
    ],
    outputs=gr.Textbox(label="Prediction"),
    title="Titanic Passenger Survival Prediction"
)

demo.launch()
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py updated ")

app.py updated 


In [38]:
req = """
pandas
numpy
scikit-learn==1.6.1
gradio
joblib
"""

with open("requirements.txt", "w") as f:
    f.write(req)

print("requirements.txt fixed ")

requirements.txt fixed 


In [39]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "Imran-Nayem/Titanic-Passanger-survial-Prediction"

api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=repo_id,
    repo_type="space"
)

api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=repo_id,
    repo_type="space"
)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/spaces/Imran-Nayem/Titanic-Passanger-survial-Prediction/commit/5907c30247994f535799af086a1f0c7362d3a6bd', commit_message='Upload requirements.txt with huggingface_hub', commit_description='', oid='5907c30247994f535799af086a1f0c7362d3a6bd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/Imran-Nayem/Titanic-Passanger-survial-Prediction', endpoint='https://huggingface.co', repo_type='space', repo_id='Imran-Nayem/Titanic-Passanger-survial-Prediction'), pr_revision=None, pr_num=None)